# Item Search Across Collections

This notebook shows the use of pystac-client to search more than one Collection in
a single request, which the STAC API spec calls "cross-collection search".

Cross-collection search is part of the core Item Search spec
so any STAC API that supports `/search` supports it.

A `/search` request normally names a single Collection, and the API returns Items from
that Collection only. Cross-collection search is the same `/search` request
with more than one Collection ID.

Cross-collection search can be an antipattern. Collections exist because their Items
differ, so mixing them in one result set is often harder to work with than searching
each Collection separately. It works best when the Collections are similar, or when the
search only uses parameters they all share. The property filter section below shows what
can go wrong.

# Client

We first connect to an API by retrieving the root catalog, or landing page, of the
API with the `Client.open` function.

In [ ]:
from collections import Counter

from pystac_client import Client

URL = "https://stac.dataspace.copernicus.eu/v1/"

client = Client.open(URL)

# AOI around Delfzijl, in northern Netherlands
aoi = {
    "type": "Polygon",
    "coordinates": [
        [[6.42, 53.17], [7.34, 53.17], [7.34, 53.67], [6.42, 53.67], [6.42, 53.17]]
    ],
}

DATETIME = "2024-06-01/2024-06-15"

# Collections as a List of IDs

The `collections` parameter accepts a list. Passing more than one Collection ID
searches all of them in a single request, rather than one request per Collection.

In [ ]:
search = client.search(
    max_items=100,
    collections=["sentinel-2-l2a", "sentinel-1-grd"],
    intersects=aoi,
    datetime=DATETIME,
)

items = list(search.items())

print(f"Collections as a list, found {len(items)} items")

# Identifying the Collection of Each Item

The matching Items are returned interleaved in a single result set, ordered by the
API rather than grouped by Collection. Use `item.collection_id` to tell them apart.

In [ ]:
counts = Counter(item.collection_id for item in items)

for collection_id, count in sorted(counts.items()):
    print(f"{collection_id}: {count}")

# Property Filters and Missing Properties

One behaviour to be aware of when combining Collections: a property filter is only
matched by Items that actually have that property. Items missing it are dropped
rather than treated as zero.

Searching an optical and a SAR Collection together while filtering on
`eo:cloud_cover` therefore removes every SAR Item, because SAR data has no such
property. The search below returns no Sentinel-1 Items at all.

In [ ]:
collections = ["sentinel-2-l2a", "sentinel-1-grd"]

unfiltered = client.search(
    max_items=200, collections=collections, intersects=aoi, datetime=DATETIME
)
filtered = client.search(
    max_items=200,
    collections=collections,
    intersects=aoi,
    datetime=DATETIME,
    query={"eo:cloud_cover": {"lt": 90}},
)

print("without a cloud filter:", Counter(i.collection_id for i in unfiltered.items()))
print("with a cloud filter:   ", Counter(i.collection_id for i in filtered.items()))

No error is raised, so this is easy to miss. When a filter applies to only some of
the Collections being searched, search them separately and combine the results.

In [ ]:
optical = client.search(
    max_items=100,
    collections=["sentinel-2-l2a"],
    intersects=aoi,
    datetime=DATETIME,
    query={"eo:cloud_cover": {"lt": 90}},
)
radar = client.search(
    max_items=100, collections=["sentinel-1-grd"], intersects=aoi, datetime=DATETIME
)

combined = list(optical.items()) + list(radar.items())

print(f"combined, found {len(combined)} items")
print(Counter(item.collection_id for item in combined))